# H-A7 — The learning gate: can a promotion rule composed from existing harnesses tell a forgetting trunk update from a clean one?

**The claim.** Weight updates from live data enter the trunk only through a nightly,
pre-registered promotion rule whose every term is an existing measurement
(`CUBBYLLM_HYPOTHESES.md` H-A7, added 2026-08-28 from the oracle-competition scoring).
This notebook runs `validation/exp_a7_learning_gate.py` on an A100: the CPU shape took
~50 min and was killed twice by session teardowns; here the whole thing is a few minutes
and the update budget is a realistic nightly one (200 steps × B32 × S512 ≈ 3.3M tokens per delta).

**Three deltas on `hd5_mem21.pt`** (151M hybrid + episodic memory, step 92k), all scored on
the *identical* held-out windows:

| delta | what | must |
|---|---|---|
| **C** | the null delta (no update) | PASS |
| **A** | a deliberately-forgetting update: fine-tune on ONE source (`nemotron_code`), no replay | be REJECTED |
| **B** | the matched clean update: same steps / LR on the full weighted mixture (= replay) | PASS |

**The gate** (thresholds pre-registered from the *paired* noise, not tuned): a term fires when the
delta raises a metric by more than max(min ε, 3 × paired SE over the shared windows) —
mix CE (ε ≥ 0.01), every source's CE (ε ≥ 0.03, names the worst), GSM8K-text CE (H-G4, ε ≥ 0.02) —
or when the H-B6 retrieval probe drops below max(0.50, base − ε_ret).

**Kill (H-A7):** the gate is not a gate if it passes A, or rejects B or C. Second kill: a full
gate pass that exceeds a nightly budget must be tiered.

**First finding, before any number:** of the six terms the competition proposed, three cannot
see a trunk delta at all — claimed-answer precision and routing precision are model-free surfaces
(word table + VM), and the NYT-vs-NLMS forgetting curve benchmarks memory *methods*. The script
reports them as *not applicable* and uses per-source CE as the trunk's own forgetting signal.

In [ ]:
# --- setup: clone repo, deps, mount Drive (run once per session) ---
import os, subprocess, sys, time
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!pip -q install torch numpy sentencepiece tokenizers
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/CubbyLLM'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- EDIT to your Drive locations; stage the token cache to local SSD ---
DRIVE     = '/content/drive/MyDrive/cubbyllm'
TOKENIZER = f'{DRIVE}/grillcheese_bbpe128k.json'
BASE_CKPT = f'{DRIVE}/hd5_mem21.pt'                 # the trained 151M hybrid+mem, step 92k
SOURCES   = f'{DRIVE}/corpus_sources.json'          # the pretrain mixture (13 sources, weights)
GSM8K     = f'{DRIVE}/socratic_test_text.jsonl'     # H-G4: the GSM8K test text (copied 2026-08-30)
CORPUS_DRIVE = f'{DRIVE}/token_cache'               # full uint32 cache (what the P6/P7 pilots used)
CACHE_MIRROR = f'{DRIVE}/token_cache_wiki'          # fallback: the mirrored Wikipedia cache
CORPUS    = '/content/token_cache'                  # LOCAL SSD (Drive-FUSE random reads crawl)
import os, glob
os.makedirs(CORPUS, exist_ok=True)
if glob.glob(f'{CORPUS_DRIVE}/*.u32'):
    !rsync -a --info=progress2 {CORPUS_DRIVE}/ {CORPUS}/
    print('staged the full token cache')
elif glob.glob(f'{CACHE_MIRROR}/*.u32'):
    !rsync -a --info=progress2 {CACHE_MIRROR}/ {CORPUS}/
    print('staged the mirrored Wikipedia cache — NOTE: not the pretrain mixture; the script will')
    print('fall back to the shards present and you must set CB_A7_SLICE to one of them')
else:
    raise SystemExit('no token cache on Drive: this experiment needs *.u32 shards')
!du -sh {CORPUS}; ls {CORPUS} | head -20
for f in (TOKENIZER, BASE_CKPT, SOURCES, GSM8K):
    print(('ok      ' if os.path.exists(f) else 'MISSING ') + f)

In [ ]:
# --- run a script, tee to a log, catch a hung child on interrupt ---
def run(script, env_extra, log_name):
    env = dict(os.environ, CUBBY_SPM=TOKENIZER, CB_CORPUS=CORPUS, **env_extra)
    os.makedirs(f'{REPO}/validation/logs', exist_ok=True)
    log = f'{REPO}/validation/logs/{log_name}'
    p = None
    try:
        with open(log, 'a', encoding='utf-8', buffering=1) as f:
            f.write(f"\n=== {time.strftime('%F %T')} "
                    + ' '.join(f'{k}={v}' for k, v in sorted(env_extra.items())) + '\n')
            p = subprocess.Popen([sys.executable, '-u', f'validation/{script}'],
                                 cwd=REPO, env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                if not line.startswith('[OK]'):      # grilly's Vulkan banner
                    print(line, end=''); f.write(line)
            p.wait()
    finally:
        if p and p.poll() is None:      # never leave a child holding the GPU
            p.terminate()

In [ ]:
# --- the experiment: GPU shape (the CPU defaults in the script are B4 / 120 steps) ---
LOGS = f'{REPO}/validation/logs'
CFG = dict(
    CB_CKPT=BASE_CKPT, CB_SOURCES=SOURCES, CB_GSM8K=GSM8K,
    CB_DEVICE='cuda',
    CB_B='32', CB_S='512',
    CB_A7_STEPS='200', CB_A7_LR='1e-4', CB_A7_WARMUP='10',   # 200 x 32 x 512 = 3.3M tokens per delta
    CB_A7_SLICE='nemotron_code',                             # the forgetting delta's single source
    CB_A7_EVAL_BATCHES='32', CB_A7_SRC_BATCHES='8', CB_A7_GSM_BATCHES='8',   # paired windows: 1024 / 256 per source / 256
    CB_A7_SAVE='1', CB_A7_SAVE_DIR=DRIVE,                    # a7_forget_nemotron_code.pt + a7_replay.pt -> Drive (~600 MB each)
    CB_A7_TAG='',
)
run('exp_a7_learning_gate.py', CFG, 'exp_a7_learning_gate.log')

In [ ]:
# --- READ + keep the evidence on Drive ---
import json, shutil
R = json.load(open(f'{LOGS}/exp_a7_learning_gate.json'))
print('verdict:', R['verdict'])
print(f"base: step {R['base']['step']} | ce_mix {R['base']['eval_seedA']['ce_mix']:.4f} | gsm8k {R['base']['eval_seedA']['ce_gsm8k']:.4f} | retrieval {R['base']['eval_seedA']['retrieval_acc']:.3f}")
for k, lab in (('A', 'forgetting'), ('B', 'clean'), ('C', 'null')):
    d = R['decisions'][k]; m = d['margins']
    print(f"{k} {lab:10s} {'PROMOTE' if d['promote'] else 'REJECT '} | dCE mix {m['d_ce_mix']:+.4f} (thr {m['eps_mix']:.4f})"
          f" | worst src {m['worst_source']} {m['d_ce_source_worst']:+.4f} (thr {m['eps_src']:.4f})"
          f" | gsm8k {m['d_ce_gsm8k']:+.4f} (thr {m['eps_gsm']:.4f}) | retrieval {m['retrieval_acc']:.3f} (floor {m['retrieval_floor']:.3f})"
          + (f" | fired {d['fired']}" if d['fired'] else ''))
srcs = list(R['decisions']['A']['margins']['d_ce_by_source'])
print('\nper-source dCE   A(forget) / B(clean) / threshold:')
for s in srcs:
    a = R['decisions']['A']['margins']; b = R['decisions']['B']['margins']
    print(f"  {s:16s} {a['d_ce_by_source'][s]:+.3f} / {b['d_ce_by_source'][s]:+.3f} / {a['eps_by_source'][s]:.3f}")
print(f"\nnightly budget check: one full gate evaluation = {R['base']['eval_seedA']['eval_wall_s']:.0f}s; whole run {R['total_wall_s']/60:.1f} min")
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
for f in glob.glob(f'{LOGS}/exp_a7_learning_gate*'):
    shutil.copy2(f, f'{DRIVE}/logs/')
print('copied log + json to', f'{DRIVE}/logs/')

In [ ]:
# --- DECIDER: the LR ladder. DONE 2026-08-30 — THE GATE SEPARATES: B promotes at 3e-5/1e-5, A is still rejected there.
# B at 1e-4 with the checkpoint's Adam restored is WORSE (+0.055 mix) -> it is the LR, not the optimizer.
# The gate separates iff A (code-only, no replay) STILL fires at the LR where B is neutral. Run those now.
import glob, shutil
LADDER_B = (('_lr3e-5', '3e-5', '0'), ('_lr1e-5', '1e-5', '0'), ('_lr1e-4_resume', '1e-4', '1'))   # done 2026-08-30
LADDER_A = (('_A_lr3e-5', '3e-5'), ('_A_lr1e-5', '1e-5'))                                           # the decider
RUN_B = False   # flip to True to redo the B ladder (A ladder below reruns by default)
if RUN_B:
    for tag, lr, resume in LADDER_B:
        run('exp_a7_learning_gate.py', dict(CFG, CB_A7_ARMS='B', CB_A7_LR=lr, CB_A7_RESUME_OPT=resume,
                                            CB_A7_TAG=tag, CB_A7_SAVE='0'), f'exp_a7_learning_gate{tag}.log')
for tag, lr in LADDER_A:
    run('exp_a7_learning_gate.py', dict(CFG, CB_A7_ARMS='A', CB_A7_LR=lr, CB_A7_RESUME_OPT='0',
                                        CB_A7_TAG=tag, CB_A7_SAVE='0'), f'exp_a7_learning_gate{tag}.log')
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
for f in glob.glob(f'{LOGS}/exp_a7_learning_gate_*'):
    shutil.copy2(f, f'{DRIVE}/logs/')
print('ladder logs copied to', f'{DRIVE}/logs/')
# read the ladder in one table
for f in sorted(glob.glob(f'{LOGS}/exp_a7_learning_gate_*.json')):
    R = json.load(open(f)); tag = f.split('exp_a7_learning_gate')[-1][:-5]
    for k in ('A', 'B'):
        if R['decisions'].get(k):
            m = R['decisions'][k]['margins']
            print(f"{tag:16s} {k} lr={R['config']['LR']:g} opt={R['deltas'][k]['finetune'].get('optimizer', '?'):8s} "
                  f"{'PROMOTE' if R['decisions'][k]['promote'] else 'REJECT '} mix {m['d_ce_mix']:+.4f} "
                  f"worst {m['worst_source']} {m['d_ce_source_worst']:+.4f} gsm {m['d_ce_gsm8k']:+.4f} ret {m['retrieval_acc']:.3f} "
                  f"fired={len(R['decisions'][k]['fired'])}")

### How to read

1. **C (null) must PROMOTE** — it is the same evaluation twice; every paired difference is exactly 0.
   If it rejects, the gate code is wrong, not the model.
2. **A (code-only, no replay) must be REJECTED**, and the reason should be *named*: expect
   `forgetting_source:<prose source>` — CE rising on books / wiki / fineweb while `nemotron_code`
   itself improves. If A promotes, the composed harnesses do not measure what update admission
   needs and H-A7's first kill fires: the learning gate needs its own benchmark.
3. **B (replay) must PROMOTE.** If it is rejected, look at *which* term fired and its margin: a
   rejection by a hair on one source at the 0.03 minimum ε is an ε-calibration question (the
   pre-registered minimum may be tighter than a clean 3M-token update's natural drift — record
   it, do not tune it in this run); a rejection on mix CE or GSM8K means a same-LR replay update
   genuinely regresses the trunk at this budget, which is itself a finding about the nightly LR.
4. **Retrieval** is the H-B6 binding-channel health: the collapsed arm read 0.115; base reads ~0.7–0.8.
   A drop below the relative floor on either delta says the update is eroding the VSA substrate.
5. **Budget:** one gate evaluation's wall time is the nightly cost of *checking*; the two fine-tunes
   are the cost of *proposing*. Both go on the H-A7 entry with the log links.

Record the verdict in `CUBBYLLM_HYPOTHESES.md` H-A7 (and the `docs/research/2026-08-28-oracle-competition-scored.md`
§3.2 pointer) with `validation/logs/exp_a7_learning_gate.{log,json}` copied from Drive.